# NSW house prices — Part 1: data, cleaning, feature engineering

This notebook walks through the first half of the project, in order:

1. Load the raw CSV and look at what is actually in it
2. Build the two postcode lookup tables the cleaning step needs
3. Clean the data — filters, duplicates, units, outliers
4. Check the result against a set of automated rules, and save it
5. Engineer features for modelling, with the leakage traps explained

Everything here calls the same functions the pipeline uses
(`pipelines/01_build_clean.py` and `pipelines/02_build_features.py`), so the numbers
match the report exactly.

**Two things to know before you run it**

* Put `nsw_property_data.csv` (610.9 MiB) in the **repository root**. It is not in git.
* The raw data has a handful of impossible values in it (a 2.7 billion sqm parcel, a
  $555M "house"). We do not delete them by hand — step 4 shows the gate that catches them.

> **This notebook writes nothing to disk.** It is a walkthrough of the logic, so the save
> steps are commented out. The real files are produced by the pipeline
> (`pipelines/01_build_clean.py`, `02_build_features.py`), which runs the same code.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook runs from notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.utils.config import (
    RAW_PROPERTY_CSV, POSTCODE_CSV, STATION_CSV, SETTINGS,
    OUT_CLEAN_PARQUET, OUT_POSTCODE_TRANSPORT, OUT_POSTCODE_DEVELOPMENT,
)

pd.set_option("display.width", 160)
print("repository root :", ROOT)
print("raw CSV present :", RAW_PROPERTY_CSV.exists(), "->", RAW_PROPERTY_CSV.name)

## 1. Look at the raw data first

Before cleaning anything, read a small slice and check:

* how many rows and columns there are
* which columns are numbers vs text
* how messy the text fields are (dates, postcodes, areas)

The file is 610 MiB, so we read only the first 200,000 rows here.

In [ ]:
preview = pd.read_csv(RAW_PROPERTY_CSV, nrows=200_000, dtype="string", low_memory=False)

print(f"columns ({preview.shape[1]}):")
for c in preview.columns:
    print("  ", c)

preview[["purchase_price", "area", "area_type", "contract_date",
         "post_code", "primary_purpose", "property_type"]].head()

### The columns that matter

| column | what it is | notes |
|---|---|---|
| `purchase_price` | sale price in AUD | read as text, because it contains blanks and junk |
| `area` + `area_type` | land/building size | units are mixed: `M` = sqm, `H` = hectares |
| `contract_date` | date of sale | several formats mixed together |
| `post_code` | postcode | sometimes has a trailing `.0` |
| `primary_purpose` | RESIDENCE / VACANT LAND / ... | we keep RESIDENCE only |
| `property_type` | house / unit / ... | we keep houses only |
| `address` | street address | used to identify a *dwelling*, not as a model feature |

Note there are **no coordinates** in this file. All the distance features are built in
step 2 from separate lookup tables.

### Why we read everything as text

`area` mixes numbers with blank strings, and `contract_date` mixes formats. If we let
pandas guess the types it will silently turn bad values into `NaN` and we lose the
ability to count what was unusable. Reading as text and converting deliberately means
we can report exactly how many rows each rule removed.

In [ ]:
# How messy are the key fields? Count what fails to parse.
price = pd.to_numeric(preview["purchase_price"], errors="coerce")
area = pd.to_numeric(preview["area"], errors="coerce")
dates = pd.to_datetime(preview["contract_date"].str.strip(), format="mixed", errors="coerce")

print(f"rows                          : {len(preview):,}")
print(f"purchase_price unparseable    : {int(price.isna().sum()):,}")
print(f"area unparseable              : {int(area.isna().sum()):,}")
print(f"contract_date unparseable     : {int(dates.isna().sum()):,}")
print()
print("area_type values:", preview["area_type"].str.strip().value_counts(dropna=False).to_dict())

## 2. Build the two postcode lookup tables

The transaction table has no coordinates and no notion of "is this a new development
area?". We build both from separate public files:

1. **Development labels** — from the transaction file itself. For a baseline period we
   compute the share of sales that were *vacant land*. A high share means the area was
   still being built out → `Greenfield`; otherwise `Established`.

2. **Transport distances** — great-circle distance from each postcode centroid to the
   CBD, to the nearest train station, and to the nearest Metro station. These use
   `australian_postcodes.csv` (centroids) and `stationentrances2020_v4.csv` (station
   coordinates). If those two files are missing they are downloaded automatically.

### A subtlety about the development labels

You cannot label a 2005 sale using a vacancy share measured over 2011–2014 — that is
information from the future. So we build **two versions** with different baselines:

| label | baseline period | legal to use from |
|---|---|---|
| `development_type_v2010` | 2001–2010 | 2011 |
| `development_type_v2014` | 2011–2014 | 2015 |

and expose only one combined column, `development_type`, which is blank where no
legally-timed label exists yet.

In [ ]:
from src.data.postcode_features import build_development_labels

development = build_development_labels(RAW_PROPERTY_CSV, SETTINGS.cleaning)

print(f"postcodes labelled: {len(development):,}")
for col in [c for c in development.columns if c.startswith("development_type_")]:
    print(f"  {col}: {development[col].value_counts(dropna=False).to_dict()}")

development.head()

In [ ]:
from src.data.postcode_features import build_transport_features

transport = build_transport_features()

print(f"postcode rows: {len(transport):,}")
transport[["postcode", "locality", "dist_cbd", "dist_train",
           "dist_metro", "dist_metro_new"]].head()

## 3. Clean the transactions

The cleaning run is one call. It reads the 610 MiB file in chunks (so memory stays
flat), joins the two lookups above, and applies every rule in order.

```python
from src.data.clean import build_clean
clean, report = build_clean(
    SETTINGS.cleaning,
    transport_features=transport,
    development_lookup=development,
)
```

`build_clean` is safe to run over the whole history because **it never learns anything
from the data** — every rule is either a fixed filter or a domain constant. Anything
that has to be *fitted* (outlier fences, imputation values) happens later, and only on
the training part of the data.

The next few cells open the pipeline up and show each rule on purpose-built examples,
so you can see what it does before trusting it.

### 3a. Required fields, then purpose and property type

Three filters in a row, exactly as in the original analysis:

1. **Required fields present** — price, area, area unit, contract date, postcode, distance
2. **Purpose is RESIDENCE** — drops vacant land and commercial sales
3. **Property type is house** — drops units, townhouses, etc.

Contract dates outside 2001-01-01 .. 2023-12-31 are also dropped: the raw file contains
typos like `0015-08-29` and `1024-01-05`.

In [ ]:
# A tiny hand-made example, so the rules are visible rather than buried in a pipeline.
demo = pd.DataFrame({
    "purchase_price": ["500000", "NaN", "300000", "800000"],
    "area":          ["600",    "500",  "400",    "700"],
    "area_type":     ["M",      "M",    "M",      "M"],
    "contract_date": ["2015-06-01", "2015-06-01", "0015-08-29", "2015-06-01"],
    "post_code":     ["2000", "2000", "2000", "2000"],
    "primary_purpose": ["RESIDENCE", "RESIDENCE", "RESIDENCE", "VACANT LAND"],
    "property_type":   ["house", "house", "house", "house"],
    "dist_cbd":      ["5.0", "5.0", "5.0", "5.0"],
})

price = pd.to_numeric(demo["purchase_price"], errors="coerce")
date = pd.to_datetime(demo["contract_date"], errors="coerce")
in_window = date.between("2001-01-01", "2023-12-31")

print("row-by-row verdict")
for i in range(len(demo)):
    reasons = []
    if pd.isna(price[i]):                       reasons.append("price missing")
    if not in_window[i]:                        reasons.append("date outside window")
    if demo.primary_purpose[i] != "RESIDENCE":  reasons.append("not RESIDENCE")
    print(f"  row {i}: {'KEEP' if not reasons else 'drop (' + ', '.join(reasons) + ')'}")

### 3b. Duplicates: compare *sale events*, not whole rows

The original analysis used `drop_duplicates()` on the whole row. That is the wrong
instrument here, and the reason is subtle:

> Later steps **drop columns** (`settlement_date`, `legal_description`,
> `strata_lot_number`). Two rows that differ only in those columns survive the
> de-duplication, and then become byte-identical in the modelling frame — where they
> silently count one transaction twice.

So we de-duplicate on an explicit key that identifies one **sale event**:

```python
DUPLICATE_KEY = ("property_id", "contract_date", "purchase_price", "area")
```

The cell below shows the difference on a small example, then runs it for real.

In [ ]:
from src.data.clean import DUPLICATE_KEY, drop_duplicate_rows

# Two rows that are the same sale, recorded twice with a different settlement date.
same_sale = pd.DataFrame([
    {"property_id": "1", "contract_date": pd.Timestamp("2015-06-01"),
     "purchase_price": 500000.0, "area": 600.0, "settlement_date": "2015-08-01"},
    {"property_id": "1", "contract_date": pd.Timestamp("2015-06-01"),
     "purchase_price": 500000.0, "area": 600.0, "settlement_date": "2015-09-01"},
])

print("whole-row de-duplication removes:", drop_duplicate_rows(same_sale)[1], "row(s)")
print("sale-event de-duplication removes:", drop_duplicate_rows(same_sale, subset=DUPLICATE_KEY)[1], "row(s)")

# And the safety check: the same property on the same day at a DIFFERENT price is
# two genuine records and must be kept.
different_price = pd.DataFrame([
    {"property_id": "2", "contract_date": pd.Timestamp("2016-03-10"),
     "purchase_price": 248680.0, "area": 500.0},
    {"property_id": "2", "contract_date": pd.Timestamp("2016-03-10"),
     "purchase_price": 348680.0, "area": 500.0},
])
print("genuine concurrent sales kept:", len(different_price) - drop_duplicate_rows(
    different_price, subset=DUPLICATE_KEY)[1], "rows")

### 3c. Units, plausibility, and the grouping key

**Area units.** `area_type` is `M` (sqm) or `H` (hectares). Hectares are multiplied by
10,000.

**Absolute plausibility gate.** This is the layer that removes *corrupted* values. It
uses domain constants, not statistics — a house is not 20 sqm and not 2.7 billion sqm:

| rule | range |
|---|---|
| area | 20 – 10,000 sqm |
| price | $10,000 – $50,000,000 |
| price per sqm | $10 – $50,000 |

The gate costs about 6.6% of rows. That is a real cost and it is reported, not hidden.

**The grouping key.** Cross-validation must not let one *dwelling* appear on both sides
of a split. We identify a dwelling by `address|postcode`. But 1.1% of addresses have no
house number at all (they are bare street names like `' WOODBURY PARK DR, MARDY'`), and
those collapse many different houses into one key — so for those rows we fall back to
`property_id`.

In [ ]:
from src.data.clean import convert_area_units, add_group_key, has_house_number

units = pd.DataFrame({
    "area": [600.0, 0.06, 0.15],
    "area_type": ["M", "H", "H"],
})
print(convert_area_units(units, SETTINGS.cleaning)[["area", "area_type", "area_sqm"]])

# Grouping key: numbered address vs bare street name
keys = pd.DataFrame({
    "address": ["12 SMITH ST", " WOODBURY PARK DR", " WOODBURY PARK DR"],
    "post_code": ["2000", "2259", "2259"],
    "property_id": ["1001", "9001", "9002"],
})
print()
print(add_group_key(keys, verbose=False)[["address", "group_key", "group_key_source"]])
print()
print("has_house_number:")
for a in ["12 SMITH ST", "UNIT 5/12 SMITH ST", " SMITH ST", "WOODBURY PARK DR"]:
    print(f"  {a!r:24s} -> {bool(has_house_number(pd.Series([a], dtype='string')).iloc[0])}")

### 3d. Outliers — three layers, and why one is not enough

This is the part worth reading carefully.

**The original analysis used a single log-IQR fence over the whole sample.** That fails
here for two reasons:

1. The extremes are *corrupted*, not merely unusual. They inflate the IQR, which
   **widens** the fence that was supposed to catch them. A single 2.7e9 sqm parcel makes
   the area fence useless.
2. A fence fitted over 2001–2015 tops out around **$2.34M**. Applied to 2023, whose
   99th percentile is **$2.9M**, it rejects legitimate recent sales.

So there are **three** layers:

| layer | what it does | fitted on |
|---|---|---|
| A — plausibility gate | removes impossible values outright | domain constants |
| B — corruption fence | removes values that cannot be a house transaction | training span |
| C — per-year quality fence | **flags** (does not delete) sample extremes | per year, whole sample |

**Why layer C only flags.** If it deleted rows, the validation windows after 2015 would
be emptied and cross-validation would be measuring nothing. So it writes an `iqr_keep`
column, and the *training* step filters on it — the validation rows stay in place.

In [ ]:
from src.data.outliers import fit_iqr_bounds, fit_global_bounds, apply_iqr_filter

pre_outlier = clean  # what build_clean returned, before the fences

train_span = pre_outlier["contract_date"].le(pd.Timestamp(SETTINGS.windows.train_end))

# Layer B: a single corruption fence, fitted on the training span only.
corruption = fit_global_bounds(pre_outlier.loc[train_span], multiplier=SETTINGS.cleaning.iqr_multiplier)
for col, (lo, hi) in corruption.items():
    print(f"corruption fence {col:16s} {lo:>12,.1f} .. {hi:>14,.1f}")

# Layer C: one quality fence per contract year.
yearly = fit_iqr_bounds(pre_outlier, by=("year",), multiplier=SETTINGS.cleaning.iqr_multiplier)

fenced, outlier_report = apply_iqr_filter(pre_outlier, yearly, global_bounds=corruption)
print()
for k, v in outlier_report.items():
    print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")

In [ ]:
# The per-year ceilings are what stop a stale fence rejecting recent sales.
bounds_frame = yearly.to_frame()
per_year = (bounds_frame[bounds_frame.column == "purchase_price"]
            .assign(group=lambda d: pd.to_numeric(d["group"], errors="coerce"))
            .dropna(subset=["group"])
            .set_index("group")["value"])

print("price fence (AUD) by year — note how the ceiling rises over time:")
for year in (2001, 2008, 2015, 2021, 2023):
    print(f"  {int(year)}: upper = ${per_year.loc[year]:,.0f}")
print()
print("A single fence fitted on 2001-2015 would sit near $2.34M and reject "
      "legitimate 2021-2023 sales. That was the bug this design fixes.")

## 4. Check the cleaning

Rules are only useful if they fail loudly. `check_clean` asserts a set of properties
(no leftover duplicates on the sale key, price and area positive, area within the
plausibility range, the grouping key present). If any rule fails, `assert_all` raises —
which is why the pipeline stops rather than writing a broken file.

In [ ]:
from src.data.contract import check_clean, assert_all, postcode_coverage

checks = check_clean(fenced)
for c in checks:
    print(c)

assert_all(checks)
print()
print("coverage:", postcode_coverage(fenced))

In [ ]:
# What did each stage remove? This is the table to read when the row count surprises you.
print(report.summary())

In [ ]:
# Saved by the pipeline, NOT by this notebook:
#
#   fenced.to_parquet(OUT_CLEAN_PARQUET, index=False)
#
# This notebook is for reading and understanding the logic, so it deliberately writes
# nothing to disk. To produce the real file, run the pipeline, which does the same work
# and additionally writes a .meta.json sidecar recording every count:
#
#   python pipelines/01_build_clean.py

print(f"clean frame in memory: {len(fenced):,} rows x {fenced.shape[1]} columns")
print(f"the pipeline would write this to {OUT_CLEAN_PARQUET.name}")

**Reference numbers from the run used for the report.** If your numbers differ, the most
likely reason is a different version of the raw extract.

| stage | rows |
|---|---|
| raw CSV | 4,854,814 |
| after house/purpose filter | 2,302,270 |
| removed as duplicate sale events | 68,380 |
| removed by the plausibility gate | 147,519 |
| removed by the corruption fence | 365,531 |
| flagged by the per-year fence (kept, training-side only) | 21,728 |
| **`clean.parquet`** | **1,720,833** |

# Part 1b — Feature engineering

`pipelines/02_build_features.py` turns the cleaned transactions into the modelling
frame. One call:

```python
frame, info = build_features(clean)      # the pipeline saves the result; this notebook does not
```

Everything in it is safe to run over the whole history, because **nothing here learns a
statistic from the target**. Imputation values, bin edges, category encodings and
scaling all happen later, inside each cross-validation fold. That split is the single
most important idea in this half of the project, and we come back to it at the end.

In [ ]:
from src.features.pipeline import build_features

frame, info = build_features(clean=None, macro=None, save=False, verbose=True)

print()
print(f"modelling frame: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
print(f"dropped for required features: {info['dropped_for_required_features']:,}")

## 5. Macro features — join "as of" the contract date, not by year

The original analysis attached a **calendar-year mean cash rate** and a hard-coded annual
CPI to every contract. That leaks information:

> A sale in January 2019 was given a rate that embeds the RBA's December 2019 decision.
> The model knew in January what happened in December.

We replace it with an **as-of join**:

* **Cash rate** — the RBA publishes each change with an effective date, so the value
  available on day *t* is the most recent change with `effective_date <= t`.
* **CPI** — the quarterly index is published about **4 weeks after** the quarter ends,
  so the value available on *t* is the most recent quarter whose
  `quarter_end + 28 days <= t`. Using `quarter_end <= t` would be wrong for the first
  four weeks of every quarter.

Snapshots are cached under `data/external/` so the run works offline.

In [ ]:
# What the as-of join actually produced. These columns exist because build_features
# already applied the join above; we just look at the result.
from src.data.macro_asof import macro_coverage_report

cols = ["contract_date", "cash_rate_asof", "cpi_index_asof",
        "cpi_asof_release_date", "cpi_yoy_asof"]
frame[cols].head(8)

In [ ]:
# Sanity check: the rate attached to a contract must never come from a later change.
sample = frame[cols].dropna(subset=["cpi_asof_release_date"]).head(2000)
never_future = (sample["cpi_asof_release_date"] <= sample["contract_date"]).all()
print("CPI value always published on or before the contract date:", bool(never_future))

## 6. Calendar and geometry features

| feature | why |
|---|---|
| `year_num` | a continuous time trend |
| `month_sin`, `month_cos` | month of year as a cycle, so December and January are treated as neighbours rather than 11 apart |
| `log_area` | price scales roughly with the log of size |
| `log_dist_cbd` | distance effects are multiplicative |
| `dist_cbd_x_area` | an interaction: a large block close to the CBD is a different product from a large block far out |

In [ ]:
from src.features.pipeline import add_calendar_features, add_geometry_features, add_target

frame = add_calendar_features(frame)
frame = add_geometry_features(frame)
frame = add_target(frame)   # log10(price) — the target

frame[["contract_date", "year_num", "month_sin", "month_cos",
       "area_sqm", "log_area", "dist_cbd", "log_dist_cbd", "purchase_price", "log_price"]].head()

### Why the target is `log10(price)`

Prices are heavily right-skewed: most sales are $500k–$2M with a thin tail of very
expensive properties. A model trained on the raw dollar price is dominated by that tail,
because squared error cares about dollars. Taking logs makes the errors **relative** —
being 10% wrong on a $250k house and on a $4M house cost the same. That is also why the
headline metric is RMSLE (root mean squared **log** error).

## 7. Neighbourhood features — the leakage-critical part

For each postcode we build a monthly panel and compute rolling statistics: the median
price over the last 3, 6 and 12 months, the median price per sqm, how many sales, how
dispersed prices were, and how long since the last sale.

**A feature for month *t* must not use any sale from month *t*.** The fix is one line,
and its position matters:

```python
lagged = panel.groupby("postcode")[cols].shift(1)   # first drop month t
roll   = lagged.groupby("postcode").rolling(12)     # then roll over t-1 and earlier
```

If you roll first and shift afterwards, or forget the shift entirely, each transaction
uses its own price to predict itself. That is **target leakage**, and it produces
excellent-looking scores that mean nothing.

Two further details:

* The panel is reindexed onto a **complete monthly grid**, so a missing month becomes an
  explicit gap. A rolling "12-month median" therefore always spans 12 calendar months.
* Windows require **full coverage** (`min_periods == 12`). The earlier version allowed a
  "12-month median" from as few as 6 observed months, which overstated the evidence
  behind the number.

In [ ]:
from src.features.panel import add_rolling_features, complete_monthly_spine, build_postcode_month_panel

one_postcode = clean[clean.post_code == clean.post_code.value_counts().index[0]]

base = build_postcode_month_panel(one_postcode)
spine = complete_monthly_spine(base)
rolling = add_rolling_features(spine)

print(f"postcode {spine['postcode'].iloc[0]}: {len(spine)} calendar months")
print()
rolling[["postcode", "month", "pc_med_price_12m", "pc_n_sales_12m",
         "pc_n_months_observed_12m", "pc_months_since_sale"]].tail(6)

In [ ]:
# Demonstrate the shift: the 12-month median for month t must exclude month t itself.
demo = spine[["postcode", "month", "median_price"]].copy()
demo["rolled_without_shift"] = (demo.groupby("postcode")["median_price"]
                                .rolling(12, min_periods=12).median()
                                .reset_index(level=0, drop=True))
demo["rolled_with_shift"] = (demo.groupby("postcode")["median_price"].shift(1)
                             .rolling(12, min_periods=12).median()
                             .reset_index(level=0, drop=True))

# On a month that had sales, the un-shifted value includes them; the shifted one cannot.
had_sales = demo.dropna(subset=["median_price"]).dropna(subset=["rolled_with_shift"]).tail(3)
had_sales

In [ ]:
# Attach the panel features to every transaction, and finish the frame.
from src.features.panel import attach_panel_features
from src.features.pipeline import clean_zoning

frame = attach_panel_features(frame, spine)
frame = attach_panel_features(frame, rolling)
frame["zoning_clean"] = clean_zoning(frame["zoning"])

pc_cols = [c for c in frame.columns if c.startswith("pc_")]
print(f"{len(pc_cols)} neighbourhood features attached:")
for c in pc_cols:
    print("  ", c)

### The two sets of panel columns, and why one is prefixed `contemporaneous_`

`attach_panel_features` returns two groups:

* **`pc_*`** — lagged rolling statistics. **These are the model features.** Safe.
* **`contemporaneous_*`** — aggregates for the *same month* as the contract, including
  other people's sales that month. Useful for describing the market, **never** allowed
  into a model.

They are kept apart explicitly, and the prefix is the warning label.

In [ ]:
contemp = [c for c in frame.columns if c.startswith("contemporaneous_")]
print("contemporaneous columns (EDA only, never a feature):")
for c in contemp:
    print("  ", c)

## 8. Final frame

The feature set falls into five groups. Part 2 uses this grouping for the ablation study
that asks which groups actually earn their place.

In [ ]:
groups = {
    "property & location": ["area_sqm", "log_area", "dist_cbd", "log_dist_cbd", "dist_train", "dist_metro"],
    "calendar":            ["year_num", "month_sin", "month_cos"],
    "labels":              ["development_type", "zoning_clean", "post_code", "council_name", "locality"],
    "lagged neighbourhood": [c for c in frame.columns if c.startswith("pc_med_price_")],
    "unit price & liquidity": [c for c in frame.columns if c.startswith(("pc_med_unit_price_", "pc_n_sales_", "pc_price_iqr_", "pc_n_months_"))],
}
for name, cols in groups.items():
    print(f"{name:24s} {len(cols):2d}  {', '.join(cols[:4])}{' ...' if len(cols) > 4 else ''}")

print()
print(f"total columns: {frame.shape[1]}")

## 9. The one rule that keeps all of this honest

Everything above is **deterministic and target-free**, so it is safe to compute once for
the whole history.

But some preprocessing genuinely needs to look at data:

* median values for imputation
* category frequencies
* target encoding
* area bin edges
* standardisation (mean / std)

If any of those are computed on the full sample, information from the validation period
leaks into training. **Preprocessing leakage** is easy to miss because the code looks
innocent:

```python
# WRONG — the median is computed using validation rows
median_area = full_frame["area_sqm"].median()

# RIGHT — computed from the training fold only, then applied to the validation fold
median_area = train_fold["area_sqm"].median()
```

All of it lives in `src/models/features.py` (`PropertyFeatureTransformer`) and is fitted
**inside each fold**, which is what Part 2 shows.

## What Part 2 covers

* Hypothesis tests: H1 (area vs price), H2 (unit price vs area) and a placebo that shows
  most of H2 is a ratio artefact
* A difference-in-differences event study of the 2019 Sydney Metro opening
* Rolling-origin cross-validation of four model families, plus a final holdout
* An ablation that asks which feature groups actually help

**Next:** `notebooks/02_analysis_and_results.ipynb`